# Mapping the Potential Destructive Power of Wildfires Using Machine Learning

## Appendix E: *NDVI Extraction*
- *Version Number: 4.0*
---
### Contents  
> 1. *Download nc files* 
> 2. *Clip and Extract California Rasters*
> 3. *Clip and Extract Oregon Rasters*
---
### Notes
- US complete daily raster data from 01/01/2018 to 12/10/2025
- Rasters obtianed from **NOAA Climate Data Record (CDR) of VIIRS Normalized Difference Vegetation Index (NDVI), Version 1**

>   Source: Vermote, Eric; NOAA CDR Program. (2022): NOAA Climate Data Record (CDR) of VIIRS Normalized Difference Vegetation Index (NDVI), Version 1. [indicate subset used]. NOAA National Centers for Environmental Information. https://doi.org/10.25921/gakh-st76. Accessed [date].
---
### Inputs
---
### Outputs  
---
### User Created Dependencies  
---

### Third Party Dependencies  
---

In [3]:
import os
import pandas as pd
import geopandas as gpd
import xarray as xr
import requests
from tqdm import tqdm
from bs4 import BeautifulSoup
import rioxarray

## File Details

In [4]:
DOWNLOAD_DIR = "ndvi_raw"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

POINTS_FILE = "../data/raw/points.gpkg"
OUTPUT_CSV = "../data/raw/ndvi_point_extraction.csv"

# LOAD POINTS

points_gdf = gpd.read_file(POINTS_FILE)

# Ensure required columns exist
assert "Sample_ID" in points_gdf.columns, "Missing Sample_ID column"
assert "Date" in points_gdf.columns, "Missing Date column"
assert "geometry" in points_gdf.columns, "Missing geometry column"

# Extract lat/lon
points_gdf["lon"] = points_gdf.geometry.x
points_gdf["lat"] = points_gdf.geometry.y

# Convert Date to datetime
points_gdf["Date"] = pd.to_datetime(points_gdf["Date"])

## Download

In [ ]:
years = ['2018','2019','2020','2021','2022','2023','2024','2025']

In [ ]:
site_address = "https://www.ncei.noaa.gov/data/land-normalized-difference-vegetation-index/access/2025/"
download_folder = "ndvi_raw"

os.makedirs(download_folder, exist_ok=True)

# Get the HTML of the directory listing
response = requests.get(site_address)
soup = BeautifulSoup(response.text, 'html.parser')

# Find all links ending with .nc
files = [a['href'] for a in soup.find_all('a', href=True) if a['href'].endswith('.nc')]

# Download each file, skip if already exists
for file in files:
    local_path = os.path.join(download_folder, file)
    
    if os.path.exists(local_path):
        print(f"Skipping {file}, already downloaded.")
        continue
    
    file_url = site_address + file
    print(f"Downloading {file}...")
    r = requests.get(file_url)
    with open(local_path, 'wb') as f:
        f.write(r.content)

print("All files processed!")


## Clip to California and Extract

In [5]:
# Input + output
input_folder = "ndvi_raw"
output_folder = "ndvi_clipped_2025"
os.makedirs(output_folder, exist_ok=True)

# Load CA boundary
ca = gpd.read_file("../data/raw/CA_counties.shp")
ca = ca.to_crs("EPSG:4326")

for fname in os.listdir(input_folder):
    if not fname.endswith(".nc"):
        continue

    fpath = os.path.join(input_folder, fname)
    print("Processing:", fname)

    ds = xr.open_dataset(fpath)

    # Extract NDVI
    ndvi = ds["NDVI"]

    # 🔧 FIX: remove encoding metadata that breaks raster export
    ndvi.attrs.pop("grid_mapping", None)
    ndvi.encoding = {}   # clears problematic CF encoding

    # Set spatial dimensions
    ndvi = (
        ndvi
        .rio.set_spatial_dims(x_dim="longitude", y_dim="latitude")
        .rio.write_crs("EPSG:4326")
    )

    # Clip to California
    clipped = ndvi.rio.clip(ca.geometry, ca.crs, drop=True)

    # Save
    out_name = fname.replace(".nc", "_CA.tif")
    out_path = os.path.join(output_folder, out_name)

    clipped.rio.to_raster(out_path)

    print("Saved:", out_name)

print("✓ All files processed.")




Processing: VIIRS-Land_v001_JP113C1_NOAA-20_20250101_c20250103153010.nc


C:\Users\dusti\anaconda3\Lib\site-packages\xarray\coding\times.py:254: RuntimeWarning: invalid value encountered in cast
  flat_num_dates_ns_int = (flat_num_dates * _NS_PER_TIME_DELTA[delta]).astype(


Saved: VIIRS-Land_v001_JP113C1_NOAA-20_20250101_c20250103153010_CA.tif
Processing: VIIRS-Land_v001_JP113C1_NOAA-20_20250102_c20250104153009.nc


C:\Users\dusti\anaconda3\Lib\site-packages\xarray\coding\times.py:254: RuntimeWarning: invalid value encountered in cast
  flat_num_dates_ns_int = (flat_num_dates * _NS_PER_TIME_DELTA[delta]).astype(


Saved: VIIRS-Land_v001_JP113C1_NOAA-20_20250102_c20250104153009_CA.tif
Processing: VIIRS-Land_v001_JP113C1_NOAA-20_20250103_c20250105153010.nc


...

Saved: VIIRS-Land_v001_JP113C1_NOAA-20_20251206_c20251208153016_CA.tif
Processing: VIIRS-Land_v001_JP113C1_NOAA-20_20251207_c20251209153020.nc


C:\Users\dusti\anaconda3\Lib\site-packages\xarray\coding\times.py:254: RuntimeWarning: invalid value encountered in cast
  flat_num_dates_ns_int = (flat_num_dates * _NS_PER_TIME_DELTA[delta]).astype(


Saved: VIIRS-Land_v001_JP113C1_NOAA-20_20251207_c20251209153020_CA.tif
Processing: VIIRS-Land_v001_JP113C1_NOAA-20_20251208_c20251210153019.nc


C:\Users\dusti\anaconda3\Lib\site-packages\xarray\coding\times.py:254: RuntimeWarning: invalid value encountered in cast
  flat_num_dates_ns_int = (flat_num_dates * _NS_PER_TIME_DELTA[delta]).astype(


Saved: VIIRS-Land_v001_JP113C1_NOAA-20_20251208_c20251210153019_CA.tif
✓ All files processed.


In [6]:
# Input + output
input_folder = "ndvi_raw"
output_folder = "ndvi_clipped_oregon_2025"
os.makedirs(output_folder, exist_ok=True)

# Load CA boundary
ca = gpd.read_file("../data/raw/OR_counties.shp")
ca = ca.to_crs("EPSG:4326")

for fname in os.listdir(input_folder):
    if not fname.endswith(".nc"):
        continue

    fpath = os.path.join(input_folder, fname)
    print("Processing:", fname)

    ds = xr.open_dataset(fpath)

    # Extract NDVI
    ndvi = ds["NDVI"]

    # 🔧 FIX: remove encoding metadata that breaks raster export
    ndvi.attrs.pop("grid_mapping", None)
    ndvi.encoding = {}   # clears problematic CF encoding

    # Set spatial dimensions
    ndvi = (
        ndvi
        .rio.set_spatial_dims(x_dim="longitude", y_dim="latitude")
        .rio.write_crs("EPSG:4326")
    )

    # Clip to California
    clipped = ndvi.rio.clip(ca.geometry, ca.crs, drop=True)

    # Save
    out_name = fname.replace(".nc", "_OR.tif")
    out_path = os.path.join(output_folder, out_name)

    clipped.rio.to_raster(out_path)

    print("Saved:", out_name)

print("✓ All files processed.")




Processing: VIIRS-Land_v001_JP113C1_NOAA-20_20250101_c20250103153010.nc


C:\Users\dusti\anaconda3\Lib\site-packages\xarray\coding\times.py:254: RuntimeWarning: invalid value encountered in cast
  flat_num_dates_ns_int = (flat_num_dates * _NS_PER_TIME_DELTA[delta]).astype(


Saved: VIIRS-Land_v001_JP113C1_NOAA-20_20250101_c20250103153010_OR.tif
Processing: VIIRS-Land_v001_JP113C1_NOAA-20_20250102_c20250104153009.nc


C:\Users\dusti\anaconda3\Lib\site-packages\xarray\coding\times.py:254: RuntimeWarning: invalid value encountered in cast
  flat_num_dates_ns_int = (flat_num_dates * _NS_PER_TIME_DELTA[delta]).astype(


Saved: VIIRS-Land_v001_JP113C1_NOAA-20_20250102_c20250104153009_OR.tif
Processing: VIIRS-Land_v001_JP113C1_NOAA-20_20250103_c20250105153010.nc


...

Saved: VIIRS-Land_v001_JP113C1_NOAA-20_20251206_c20251208153016_OR.tif
Processing: VIIRS-Land_v001_JP113C1_NOAA-20_20251207_c20251209153020.nc


C:\Users\dusti\anaconda3\Lib\site-packages\xarray\coding\times.py:254: RuntimeWarning: invalid value encountered in cast
  flat_num_dates_ns_int = (flat_num_dates * _NS_PER_TIME_DELTA[delta]).astype(


Saved: VIIRS-Land_v001_JP113C1_NOAA-20_20251207_c20251209153020_OR.tif
Processing: VIIRS-Land_v001_JP113C1_NOAA-20_20251208_c20251210153019.nc


C:\Users\dusti\anaconda3\Lib\site-packages\xarray\coding\times.py:254: RuntimeWarning: invalid value encountered in cast
  flat_num_dates_ns_int = (flat_num_dates * _NS_PER_TIME_DELTA[delta]).astype(


Saved: VIIRS-Land_v001_JP113C1_NOAA-20_20251208_c20251210153019_OR.tif
✓ All files processed.
